# EDA · Seguridad Social (afiliados, pensiones y ratio)

- No participa en producción y el pipeline no lo necesita.
- Lee las salidas verificadas de Plata mediante `SilverReader` (sha256 contra el manifiesto); no vuelve a interpretar los Excel de Bronce.
- Toda la lógica de negocio vive en `src/`; aquí solo se explora y se grafica.
- Requisito: haber ejecutado `python main.py` (o `--desde plata`) al menos una vez.

> **Anexo.** Su contenido (frecuencia de las fuentes y comparación stock de diciembre frente a media anual) está integrado en el notebook principal `notebooks/Proyecto_ETL_Pensiones_Espana.ipynb`, sección 5.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import yaml

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "config" / "config.yaml").exists())
sys.path.insert(0, str(ROOT))

from src.utils.silver_reader import SilverReader

with (ROOT / "config" / "config.yaml").open(encoding="utf-8") as file:
    config = yaml.safe_load(file)
silver = config["silver"]["seguridad_social"]


def latest(manifests_path, dataset_name):
    return SilverReader(ROOT / manifests_path, dataset_name).latest()


afiliados = latest(silver["manifests_path"], silver["afiliados"]["dataset_name"])["frame"]
pensiones = latest(silver["manifests_path"], silver["pensiones"]["dataset_name"])["frame"]
pensiones_kpi = config["indicadores"]["pensiones"]  # el ratio se calcula ahora en la capa de indicadores de Oro
kpi = latest(pensiones_kpi["manifests_path"], pensiones_kpi["dataset_name"])["frame"]

## Frecuencia real de cada fuente

- Afiliados: serie mensual completa (último día de cada mes).
- Pensiones: serie anual a diciembre (nota de la fuente: *Datos anuales a diciembre de cada año.*) más mensual reciente (día 1 de cada mes).

In [ ]:
print("Afiliados:", afiliados["fecha_referencia"].min(), "→", afiliados["fecha_referencia"].max(), f"({len(afiliados)} meses)")
display(pensiones.groupby("tipo_corte").agg(filas=("anyo", "size"), desde=("fecha_referencia", "min"), hasta=("fecha_referencia", "max")))

## Metodología original frente a la adoptada

La primera versión del EDA cruzaba la media anual de afiliados con el dato de pensiones a diciembre,
y `kpi_ratio.py` promediaba los meses disponibles: diciembre para 2016–2024 y doce meses para 2025.
Esa mezcla no es homogénea. El pipeline usa stock de diciembre / stock de diciembre en todos los años
(`metodologia_ratio = stock_diciembre`). La tabla muestra la diferencia entre ambos criterios.

In [ ]:
media_afiliados = afiliados.groupby("anyo")["total_afiliados"].mean().rename("afiliados_media_anual")
comparacion = kpi.set_index("anyo")[["total_afiliados", "total_pensiones", "ratio_cotizantes_pensionistas"]].join(media_afiliados)
comparacion["ratio_media_anual_original"] = comparacion["afiliados_media_anual"] / comparacion["total_pensiones"]
comparacion["diferencia"] = comparacion["ratio_cotizantes_pensionistas"] - comparacion["ratio_media_anual_original"]
comparacion.round(4)

In [ ]:
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(kpi["anyo"], kpi["total_afiliados"] / 1e6, label="Afiliados a diciembre (millones)", marker="o", color="blue")
plt.plot(kpi["anyo"], kpi["total_pensiones"] / 1e6, label="Pensiones a diciembre (millones)", marker="s", color="red")
plt.title("Evolución: afiliados frente a pensiones", fontsize=14)
plt.xlabel("Año")
plt.ylabel("Millones")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(kpi["anyo"], kpi["ratio_cotizantes_pensionistas"], marker="^", color="green", linewidth=2, label="stock_diciembre")
plt.plot(comparacion.index, comparacion["ratio_media_anual_original"], linestyle="--", color="gray", label="media anual (versión original)")
plt.title("KPI: ratio cotizantes / pensionistas", fontsize=14)
plt.xlabel("Año")
plt.ylabel("Ratio")
plt.legend()

plt.tight_layout()
plt.show()

kpi.tail()